# Challenge 2: Your first offline eval

An **offline eval** runs your agent over a fixed dataset and scores every output. Three pieces:

1. **Dataset:** inputs plus reference outputs.
2. **Target function:** runs your agent on one example's inputs.
3. **Evaluators:** score the target's output against the reference.

**One cell was blank in the workshop. Here it is filled in, so every cell runs.**

In [ ]:
import os
import sys

for path in ("..", "../lab"):
    if path not in sys.path:
        sys.path.insert(0, path)

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from langsmith import Client, uuid7

import lab_eval
from eval_lab.agent import SYSTEM_PROMPT_V1, TOOLS, build_agent, extract_tool_calls

client = Client()
DATASET = lab_eval.dataset_name()
print("project:", os.environ["LANGSMITH_PROJECT"])
print("dataset:", DATASET)

## The dataset

Eight examples, each with an `inputs.query` and reference `outputs`. Later challenges reuse these fields:

| Reference field | Used for |
|---|---|
| `keywords` | A cheap string check on the answer (this challenge) |
| `reference_answer` | A rubric an LLM judge reads (challenge 3) |
| `first_tool`, `order_id` | Single-step tool selection (challenge 3) |
| `trajectory` | The tool names expected, in order (challenges 4 and 5) |

Examples 4 and 7 are refund requests that must be **declined**. Their expected trajectory has no `issue_refund`.

In [ ]:
EXAMPLES = [
    {"inputs": {"query": "Where is order A1003?"},
     "outputs": {"reference_answer": "Says order A1003 has shipped but is not yet delivered.",
                 "keywords": ["shipped"], "first_tool": "lookup_order", "order_id": "A1003",
                 "trajectory": ["lookup_order"]}},
    {"inputs": {"query": "Can I return order A1001?"},
     "outputs": {"reference_answer": "Says order A1001 is eligible for return.",
                 "keywords": ["eligible"], "first_tool": "lookup_order", "order_id": "A1001",
                 "trajectory": ["lookup_order", "check_refund_eligibility"]}},
    {"inputs": {"query": "Please refund order A1001."},
     "outputs": {"reference_answer": "Issues a refund for A1001 and confirms refund REF-A1001.",
                 "keywords": ["REF-A1001"], "first_tool": "lookup_order", "order_id": "A1001",
                 "trajectory": ["lookup_order", "check_refund_eligibility", "issue_refund"]}},
    {"inputs": {"query": "Refund order A1002."},
     "outputs": {"reference_answer": "Declines the refund because A1002 was delivered over 30 days ago. No refund issued.",
                 "keywords": ["eligible"], "first_tool": "lookup_order", "order_id": "A1002",
                 "trajectory": ["lookup_order", "check_refund_eligibility"]}},
    {"inputs": {"query": "What is your return policy?"},
     "outputs": {"reference_answer": "States that items can be returned within 30 days of delivery.",
                 "keywords": ["30"], "first_tool": "search_faq", "order_id": None,
                 "trajectory": ["search_faq"]}},
    {"inputs": {"query": "How long does standard shipping take?"},
     "outputs": {"reference_answer": "States that standard shipping takes 3-5 business days.",
                 "keywords": ["3-5"], "first_tool": "search_faq", "order_id": None,
                 "trajectory": ["search_faq"]}},
    {"inputs": {"query": "Refund order A1003 please."},
     "outputs": {"reference_answer": "Declines the refund because A1003 has not been delivered. No refund issued.",
                 "keywords": ["delivered"], "first_tool": "lookup_order", "order_id": "A1003",
                 "trajectory": ["lookup_order", "check_refund_eligibility"]}},
    {"inputs": {"query": "I want my money back for A1004."},
     "outputs": {"reference_answer": "Issues a refund for A1004 and confirms refund REF-A1004.",
                 "keywords": ["REF-A1004"], "first_tool": "lookup_order", "order_id": "A1004",
                 "trajectory": ["lookup_order", "check_refund_eligibility", "issue_refund"]}},
]
print(len(EXAMPLES), "examples")

##  create the dataset

The next cell holds this code. It creates the dataset and uploads the examples, unless the dataset already exists (so rerunning is safe):

```python
if client.has_dataset(dataset_name=DATASET):
    dataset = client.read_dataset(dataset_name=DATASET)
    print(f"'{DATASET}' already exists; leaving it as is")
else:
    dataset = client.create_dataset(DATASET)
    client.create_examples(
        inputs=[e["inputs"] for e in EXAMPLES],
        outputs=[e["outputs"] for e in EXAMPLES],
        dataset_id=dataset.id,
    )
    print(f"Created '{DATASET}' with {len(EXAMPLES)} examples")
```

In [ ]:
# Solution. In the workshop this cell was blank and you wrote it.
if client.has_dataset(dataset_name=DATASET):
    dataset = client.read_dataset(dataset_name=DATASET)
    print(f"'{DATASET}' already exists; leaving it as is")
else:
    dataset = client.create_dataset(DATASET)
    client.create_examples(
        inputs=[e["inputs"] for e in EXAMPLES],
        outputs=[e["outputs"] for e in EXAMPLES],
        dataset_id=dataset.id,
    )
    print(f"Created '{DATASET}' with {len(EXAMPLES)} examples")

## The target function

A target takes one example's `inputs` and returns whatever the evaluators need. Each call builds a fresh agent run with its own thread, so examples don't affect each other.

In [ ]:
agent = build_agent()


def run_agent(inputs: dict) -> dict:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config={"configurable": {"thread_id": str(uuid7())}},
    )
    return {"response": result["messages"][-1].text}

## An evaluator

An evaluator is a function that returns `{"key": ..., "score": ...}`. This one is deliberately simple: the fraction of expected `keywords` that appear in the response.

In [ ]:
def keyword_match(outputs, reference_outputs):
    text = outputs["response"].lower()
    hits = [k for k in reference_outputs["keywords"] if k.lower() in text]
    return {"key": "keyword_match", "score": len(hits) / len(reference_outputs["keywords"])}

## Run the experiment

`client.evaluate()` runs the target on every example, applies the evaluators, and records one **experiment** in LangSmith. `experiment_prefix` names it. This takes about a minute.

In [ ]:
results = client.evaluate(
    run_agent,
    data=DATASET,
    evaluators=[keyword_match],
    experiment_prefix="first-eval",
    max_concurrency=2,
)
print("experiment:", results.experiment_name)

## Read the scores

Print the average `keyword_match` for the experiment. Then open **Datasets & Experiments** in LangSmith, open your dataset, and look at the experiment. Find the lowest-scoring row and open its trace.

In [ ]:
lab_eval.compare(prefixes=("first-eval",))

**Result:** a table with `first-eval-...`, 8 runs, and an average `keyword_match`. A string check is brittle: a correct answer worded differently scores lower. Challenge 3 adds a model-based judge.